Contents:


# Exploratory Data Analysis: EIA Metadata

## Introduction

This notebook analyzes the U.S. Energy Information Administration (EIA) electricity metadata retrieved from the v2 API. The goal is to understand how time series are structured across datasets, frequencies, and facet dimensions to support downstream data exploration and analysis.

## Imports

In [15]:
%load_ext autoreload
%autoreload 2
import logging
import os
from io import StringIO

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
import helpers.hs3 as hs3
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Configure the notebook style.
hprint.config_notebook()

# Configure S3.
s3_dir = "s3://causify-data-collaborators/causal_automl/metadata/"
aws_profile = "ck"

## Load Data

In [24]:
# Load EIA metadata index.
metadata_s3_path = f"{s3_dir}eia_electricity_metadata_original_v1.0.csv"
metadata_csv = hs3.from_file(metadata_s3_path, aws_profile=aws_profile)
df_metadata = pd.read_csv(StringIO(metadata_csv))

In [25]:
# Preview metadata.
print(df_metadata.shape)
print(df_metadata.columns)
df_metadata.head()

(172, 17)
Index(['url', 'id', 'dataset_id', 'name', 'description', 'frequency_id', 'frequency_alias', 'frequency_description', 'frequency_query', 'frequency_format', 'facets', 'data', 'data_alias', 'data_units', 'start_period', 'end_period', 'parameter_values_file'], dtype='object')


,url,id,dataset_id,name,description,frequency_id,frequency_alias,frequency_description,frequency_query,frequency_format,facets,data,data_alias,data_units,start_period,end_period,parameter_values_file
0,https://api.eia.gov/v2/electricity/retail-sale...,retail_sales_monthly_revenue,retail_sales,Electricity Sales to Ultimate Customers,Electricity sales to ultimate customer by stat...,monthly,NaN,One data point for each month.,M,YYYY-MM,"[{'id': 'stateid', 'description': 'State / Cen...",revenue,Revenue from Sales to Ultimate Customers,million dollars,2001-01,2025-02,eia_parameters_v1.0/retail_sales_parameters.csv
1,https://api.eia.gov/v2/electricity/retail-sale...,retail_sales_monthly_sales,retail_sales,Electricity Sales to Ultimate Customers,Electricity sales to ultimate customer by stat...,monthly,NaN,One data point for each month.,M,YYYY-MM,"[{'id': 'stateid', 'description': 'State / Cen...",sales,Megawatt-hours Sold to Ultimate Customers,million kilowatt hours,2001-01,2025-02,eia_parameters_v1.0/retail_sales_parameters.csv
2,https://api.eia.gov/v2/electricity/retail-sale...,retail_sales_monthly_price,retail_sales,Electricity Sales to Ultimate Customers,Electricity sales to ultimate customer by stat...,monthly,NaN,One data point for each month.,M,YYYY-MM,"[{'id': 'stateid', 'description': 'State / Cen...",price,Average Price of Electricity to Ultimate Custo...,cents per kilowatt-hour,2001-01,2025-02,eia_parameters_v1.0/retail_sales_parameters.csv
3,https://api.eia.gov/v2/electricity/retail-sale...,retail_sales_monthly_customers,retail_sales,Electricity Sales to Ultimate Customers,Electricity sales to ultimate customer by stat...,monthly,NaN,One data point for each month.,M,YYYY-MM,"[{'id': 'stateid', 'description': 'State / Cen...",customers,Number of Ultimate Customers,number of customers,2001-01,2025-02,eia_parameters_v1.0/retail_sales_parameters.csv
4,https://api.eia.gov/v2/electricity/retail-sale...,retail_sales_quarterly_revenue,retail_sales,Electricity Sales to Ultimate Customers,Electricity sales to ultimate customer by stat...,quarterly,NaN,One data point every 3 months.,Q,"YYYY-""Q""Q","[{'id': 'stateid', 'description': 'State / Cen...",revenue,Revenue from Sales to Ultimate Customers,million dollars,2001-01,2025-02,eia_parameters_v1.0/retail_sales_parameters.csv


## EIA Metadata Index Column information

| **Column**              | **Description**                                                                 |
|-------------------------|---------------------------------------------------------------------------------|
| `url`                   | Full API URL to access the time series.                                         |
| `id`                    | Unique identifier for the time series.                                          |
| `dataset_id`            | Dataset group this series belongs to.                                           |
| `name`                  | Human-readable title of the time series.                                        |
| `description`           | Full description of what the time series measures.                              |
| `frequency_id`          | Frequency label (e.g. `monthly`, `quarterly`, `hourly`).                        |
| `frequency_alias`       | Alternative frequency name (often missing).                                     |
| `frequency_description` | Sentence-style explanation of frequency (e.g. "One data point for each month"). |
| `frequency_query`       | Query shorthand for frequency (e.g. `M` for monthly).                           |
| `frequency_format`      | Formatting string used in time index.                                           |
| `facets`                | JSON-style list of dimension fields used for the series.                        |
| `data`                  | Short name of the measured value (e.g. `revenue`, `sales`).                     |
| `data_alias`            | Human-readable version of the data name.                                        |
| `data_units`            | Units of measurement (e.g. `million dollars`, `cents per kWh`).                 |
| `start_period`          | Start date of available data (YYYY-MM format).                                  |
| `end_period`            | End date of available data (YYYY-MM format).                                    |
| `parameter_values_file` | S3 path to facet value mappings for this dataset.                               |


In [26]:
# Load parameter files for EIA metadata.
param_s3_dir = f"{s3_dir}eia_parameters_v1.0"
aws_profile = "ck"
s3fs_ = hs3.get_s3fs(aws_profile)
param_paths = s3fs_.ls(param_s3_dir)

# Load each file into a dictionary of DataFrames.
param_dfs = {}
for path in param_paths:
    if path.endswith(".csv"):
        key = os.path.basename(path).replace("_parameters.csv", "")
        full_path = f"s3://{path}" if not path.startswith("s3://") else path
        csv_text = hs3.from_file(full_path, aws_profile=aws_profile)
        param_dfs[key] = pd.read_csv(StringIO(csv_text))

In [34]:
# Available parameter datasets.
print(param_dfs.keys())

# Preview one parameter file.
param_dfs["retail_sales"].head()

dict_keys(['capability', 'daily_fuel_type_data', 'daily_interchange_data', 'daily_region_data', 'daily_region_sub_ba_data', 'electric_power_operational_data', 'emissions_by_state_by_fuel', 'energy_efficiency', 'facility_fuel', 'fuel_type_data', 'interchange_data', 'meters', 'net_metering', 'operating_generator_capacity', 'region_data', 'region_sub_ba_data', 'retail_sales', 'source_disposition', 'summary'])


,dataset_id,facet_id,id,name,alias
0,retail_sales,stateid,IN,Indiana,(IN) Indiana
1,retail_sales,stateid,KS,Kansas,(KS) Kansas
2,retail_sales,stateid,MAT,Middle Atlantic,Region: (MAT) Middle Atlantic
3,retail_sales,stateid,CT,Connecticut,(CT) Connecticut
4,retail_sales,stateid,VA,Virginia,(VA) Virginia


### EIA Parameters Column information

| **Column**         | **Description**                                                                 |
|--------------------|---------------------------------------------------------------------------------|
| `dataset_id`       | Name of the parent dataset this parameter file belongs to (e.g. `retail_sales`). |
| `facet_id`         | The dimension or facet described (e.g. `stateid`, `sectorid`).                   |
| `id`               | The unique code or shorthand for the facet value (e.g. `CA`, `RES`).             |
| `name`             | Plain name of the facet value (e.g. `California`, `residential`).                |
| `alias`            | Optional display-friendly name or formatted version of the value.                |
